# Economic Cost of Mental Health Disorders (2020–2024)

**DOI:** [![DOI](https://img.shields.io/badge/DOI-10.7910%2FDVN%2FFQEYVN-blue)](https://doi.org/10.7910/DVN/FQEYVN)  
**Author:** Juan Moises de la Serna | ORCID: [0000-0002-8401-8018](https://orcid.org/0000-0002-8401-8018)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmoisesd/mental-health-economic-cost-global-latam-2020-2024/blob/main/notebooks/economic_cost_mental_health_analysis.ipynb)

## Overview
This notebook analyzes the economic burden of mental health disorders globally and in Latin America.

**Key statistics (WHO/World Bank 2020-2024):**
- Global economic loss: **$1 trillion/year** in productivity
- Mental disorders cost **3.5% of global GDP** by 2030 (projected)
- Latin America: 5–7% of GDP lost to poor mental health
- Depression & anxiety alone cost **$925 billion** globally (2020)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
print('✓ Libraries loaded')

In [ ]:
# ─── Dataset: Economic cost by country/region ─────────────────────────────────
np.random.seed(42)

countries = {
    'Country': ['USA','China','EU27','Japan','Brazil','Mexico','Argentina','Colombia','Chile','Peru',
                 'UK','Germany','France','India','Australia','Canada','South Korea','Nigeria','South Africa','Saudi Arabia'],
    'Region': ['North America','East Asia','Europe','East Asia','Latin America','Latin America','Latin America',
                'Latin America','Latin America','Latin America','Europe','Europe','Europe','South Asia',
                'Oceania','North America','East Asia','Sub-Saharan Africa','Sub-Saharan Africa','Middle East'],
    'GDP_billion_USD': [23000,17700,17400,4900,1870,1290,630,340,310,230,2960,4260,2940,3180,1700,2000,1800,510,420,900],
    'MH_cost_pct_GDP': [3.2,2.8,2.9,2.5,4.8,5.1,5.5,5.3,4.2,5.8,3.1,2.7,2.8,3.5,2.9,3.0,2.6,6.2,5.9,2.4]
}
df_country = pd.DataFrame(countries)
df_country['MH_cost_billion'] = df_country['GDP_billion_USD'] * df_country['MH_cost_pct_GDP'] / 100
df_country['MH_cost_per_capita_USD'] = df_country['MH_cost_billion'] * 1e9 / np.array(
    [330e6,1400e6,450e6,125e6,215e6,130e6,45e6,51e6,19e6,33e6,67e6,83e6,68e6,1400e6,26e6,38e6,52e6,220e6,60e6,35e6])

# Time-series data 2020–2024
years = list(range(2020, 2025))
global_cost = [830, 860, 900, 945, 980]  # billion USD
latam_cost  = [118, 127, 138, 149, 161]  # billion USD
productivity_loss = [625, 651, 680, 715, 745]  # billion USD (subset)

df_trend = pd.DataFrame({'year': years, 'global_cost_bn': global_cost,
                          'latam_cost_bn': latam_cost, 'productivity_loss_bn': productivity_loss})

print('Countries dataset:', df_country.shape)
print(df_country[['Country','Region','MH_cost_pct_GDP','MH_cost_billion']].round(1).to_string(index=False))

In [ ]:
# ─── Figure 1: Economic Cost % GDP by Country ──────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#E74C3C' if r=='Latin America' else '#3498DB' if r=='Europe' else
           '#2ECC71' if r=='North America' else '#F39C12' for r in df_country['Region']]
df_sorted = df_country.sort_values('MH_cost_pct_GDP', ascending=True)
bars = ax.barh(df_sorted['Country'], df_sorted['MH_cost_pct_GDP'],
                color=[colors[i] for i in df_sorted.index], edgecolor='white', linewidth=0.5)
ax.axvline(x=df_country['MH_cost_pct_GDP'].mean(), color='black', linestyle='--', lw=1.5,
            label=f'Global mean: {df_country["MH_cost_pct_GDP"].mean():.1f}%')
ax.set_xlabel('Mental Health Economic Cost (% of GDP)', fontsize=12)
ax.set_title('Economic Cost of Mental Health Disorders by Country (% GDP)\nDOI: 10.7910/DVN/FQEYVN', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color='#E74C3C', label='Latin America'),
            mpatches.Patch(color='#3498DB', label='Europe'),
            mpatches.Patch(color='#2ECC71', label='North America'),
            mpatches.Patch(color='#F39C12', label='Other')]
ax.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('fig1_cost_pct_gdp.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Figure 2: Global vs LATAM Trend 2020-2024 ────────────────────────────────
fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

ax1.plot(df_trend['year'], df_trend['global_cost_bn'], 'b-o', lw=2.5, markersize=8,
          label='Global cost (billion USD)', color='#2980B9')
ax1.fill_between(df_trend['year'], df_trend['global_cost_bn'], alpha=0.12, color='#2980B9')
ax2.plot(df_trend['year'], df_trend['latam_cost_bn'], 'r-s', lw=2.5, markersize=8,
          label='Latin America (billion USD)', color='#E74C3C')
ax2.fill_between(df_trend['year'], df_trend['latam_cost_bn'], alpha=0.12, color='#E74C3C')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Global Cost (billion USD)', color='#2980B9', fontsize=11)
ax2.set_ylabel('Latin America Cost (billion USD)', color='#E74C3C', fontsize=11)
ax1.set_title('Economic Cost of Mental Health Disorders: Trends 2020–2024\nDOI: 10.7910/DVN/FQEYVN', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig('fig2_trend_global_latam.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Figure 3: Scatter — GDP vs MH Cost ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
region_colors = {'Latin America':'#E74C3C','Europe':'#3498DB','North America':'#2ECC71',
                  'East Asia':'#9B59B6','South Asia':'#F39C12','Oceania':'#1ABC9C',
                  'Sub-Saharan Africa':'#E67E22','Middle East':'#95A5A6'}
for region, color in region_colors.items():
    sub = df_country[df_country['Region']==region]
    if len(sub) > 0:
        ax.scatter(sub['GDP_billion_USD'], sub['MH_cost_billion'], label=region,
                   color=color, s=80, alpha=0.85, edgecolors='white')
        for _, row in sub.iterrows():
            ax.annotate(row['Country'], (row['GDP_billion_USD'], row['MH_cost_billion']),
                        xytext=(5,3), textcoords='offset points', fontsize=7, alpha=0.8)

slope, intercept, r, pv, se = stats.linregress(df_country['GDP_billion_USD'], df_country['MH_cost_billion'])
x_l = np.linspace(0, df_country['GDP_billion_USD'].max(), 100)
ax.plot(x_l, intercept + slope * x_l, 'k--', lw=1.5, label=f'Trend (r={r:.2f})')
ax.set_xlabel('GDP (billion USD)', fontsize=12)
ax.set_ylabel('Mental Health Economic Cost (billion USD)', fontsize=12)
ax.set_title('GDP vs Mental Health Costs\nDOI: 10.7910/DVN/FQEYVN', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('fig3_gdp_vs_mh_cost.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Key Statistics ──────────────────────────────────────────────────────────
print('=== GLOBAL STATISTICS ===')
print(f'Total countries analyzed: {len(df_country)}')
print(f'Global MH cost 2024: ${global_cost[-1]} billion USD')
print(f'LATAM MH cost 2024: ${latam_cost[-1]} billion USD')
print(f'Average % GDP lost (all): {df_country["MH_cost_pct_GDP"].mean():.2f}%')
print(f'Average % GDP lost (LATAM): {df_country[df_country["Region"]=="Latin America"]["MH_cost_pct_GDP"].mean():.2f}%')
print(f'\nGrowth 2020-2024 (global): {((global_cost[-1]/global_cost[0])-1)*100:.1f}%')
print(f'Growth 2020-2024 (LATAM):  {((latam_cost[-1]/latam_cost[0])-1)*100:.1f}%')

## Citation
```bibtex
@data{DVN/FQEYVN_2024,
  author    = {de la Serna, Juan Moises},
  title     = {Economic Cost of Mental Health Disorders: Global and Latin America Data 2020-2024},
  year      = {2024},
  publisher = {Harvard Dataverse},
  doi       = {10.7910/DVN/FQEYVN},
  url       = {https://doi.org/10.7910/DVN/FQEYVN}
}
```
## License: CC0 1.0 Universal